# SmartGrid CLI Demo Notebook V3

Notebook **CLI-first**, **replay-first** et **strict day-ahead first** pour une démo complète :

- entraîner les configs officielles via `make train-consumption`,
- lancer le benchmark métier officiel via `scripts/benchmark_replay_models.py`,
- auditer clairement les `skipped_days`,
- produire un **long sample runtime** de `2025-11-20` à `2026-03-19`,
- comparer **réel vs weekly naive vs legacy vs modèle(s)**,
- exporter des artefacts propres pour une analyse ultérieure dans ChatGPT.


## A. Scope, conventions, semantics

Règles de lecture de ce notebook :

- **benchmark officiel** = `replay`, pas l'offline test ;
- **scope officiel** = `strict_day_ahead` uniquement ;
- **offline test** = diagnostic secondaire, utile mais non décisif pour le classement ;
- **long sample runtime** = vérification et démo de la chaîne métier sur une plage continue ;
- **legacy** = comparaison valide jusqu'au `2025-12-10` inclus, puis absente explicitement ;
- **promotion** = uniquement après le replay-first ranking ;
- toutes les commandes sont exécutées depuis la racine du repo avec `cwd=ROOT`.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "Makefile").exists() and (candidate / "src" / "smartgrid").exists():
            return candidate
    raise RuntimeError(
        f"Impossible de trouver la racine du repo SmartGrid depuis {start_path}. "
        "Ouvre le notebook depuis le repo ou fixe ROOT manuellement."
    )


ROOT = find_repo_root()
for candidate in [ROOT, ROOT / "src"]:
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import pandas as pd

from smartgrid.data.catalog import resolve_consumption_data_config
from smartgrid.data.loaders import load_history, load_old_benchmark
from smartgrid.evaluation.metrics import build_metrics_df, compute_metrics_v2
from smartgrid.notebooks.cli_demo_utils import (
    build_config_inventory,
    build_demo_paths,
    build_model_label_map,
    build_skipped_days_audit,
    build_truth_baseline_frame,
    build_wide_comparison_frame,
    coerce_datetime,
    collect_consumption_runs,
    configure_pandas_display,
    find_latest_replay_summary,
    load_or_run_long_sample_predictions,
    load_replay_summary,
    make_overrides,
    normalize_replay_summary,
    optional_cli_args,
    prepare_legacy_forecast_frame,
    run_cli,
    select_latest_runs_per_config,
    slice_date_range,
    slice_single_day,
    write_json,
)

configure_pandas_display()
PATHS = build_demo_paths(ROOT)
ARTIFACTS = PATHS.artifacts_root
EXPORT_ROOT = PATHS.notebook_export_root

plt.style.use("seaborn-v0_8-whitegrid")

print("ROOT:", ROOT)
print("ARTIFACTS:", ARTIFACTS)
print("NOTEBOOK_EXPORT_ROOT:", EXPORT_ROOT)


ROOT: /home/khalid/cours/m1/SmartGrid_Project/smart-grid
ARTIFACTS: /home/khalid/cours/m1/SmartGrid_Project/smart-grid/artifacts
NOTEBOOK_EXPORT_ROOT: /home/khalid/cours/m1/SmartGrid_Project/smart-grid/artifacts/notebook_exports/cli_demo_v3


## B. User Configuration

Édite principalement cette cellule pour piloter la démo.


In [2]:
# ---- Execution toggles
RUN_ENVIRONMENT_CHECKS = True
RUN_TRAINING = True
RUN_REPLAY = True
RUN_LONG_SAMPLE_PREDICT = True
RUN_PROTOCOL_AUDIT = True
RUN_PROMOTION = True

# ---- Load / cache behavior
LOAD_EXISTING_RUNS = True
LOAD_LATEST_REPLAY_IF_NOT_RUN = True
LOAD_CACHED_LONG_SAMPLE = True
FORCE_RECOMPUTE_LONG_SAMPLE = False
LONG_SAMPLE_CONTINUE_ON_ERROR = False

# ---- Official replay scope
ONLY_STRICT_DAY_AHEAD = True
REPLAY_START_DATE = "2025-11-01"
REPLAY_END_DATE = "2025-11-30"

# ---- Long runtime sample
LONG_SAMPLE_START_DATE = "2025-11-20"
LONG_SAMPLE_END_DATE = "2026-03-19"
LEGACY_COVERAGE_END_DATE = "2025-12-10"

# ---- Dataset resolution
DATASET_KEY = "full_2020_2026"
CATALOG_PATH = None
HISTORICAL_CSV = None
WEATHER_CSV = None
HOLIDAYS_XLSX = None
BENCHMARK_CSV = None
LEGACY_FORECAST_CSV = None

# ---- Official training candidates
CONFIG_PATHS = [
    "configs/consumption/mlp_strict_day_ahead_baseline.yaml",
    "configs/consumption/mlp_strict_day_ahead_weather_basic.yaml",
    "configs/consumption/mlp_strict_day_ahead_cyclical_weather_basic.yaml",
    "configs/consumption/mlp_strict_day_ahead_cyclical_weather_shifted_dynamics.yaml",
]
ANALYSIS_DAYS = 1

# ---- Replay selection / model overlays
REPLAY_MODEL_RUN_IDS = []
SELECTED_RUN_ID = None
COMPARE_RUN_IDS = []
COMPARE_TOP_K = 3
LONG_SAMPLE_RUN_IDS = []

# ---- Audit / storytelling dates
PROTOCOL_AUDIT_DATE = "2025-11-20"
SINGLE_DAY_PLOTS = [
    "2025-11-20",
    "2025-12-10",
    "2026-01-15",
    "2026-03-19",
]

# ---- Promotion
PROMOTE_REPLAY_WINNER = False


## C. Environment sanity checks

Cette section doit échouer tôt si le notebook est lancé hors contexte repo ou si les fichiers critiques manquent.


In [3]:
resolved_data_config = resolve_consumption_data_config(
    {
        "dataset_key": DATASET_KEY,
        "date_col": "Date",
        "target_name": "tot",
    },
    project_root=ROOT,
    catalog_path=CATALOG_PATH,
    overrides={
        "historical_csv": HISTORICAL_CSV,
        "weather_csv": WEATHER_CSV,
        "holidays_xlsx": HOLIDAYS_XLSX,
        "benchmark_csv": BENCHMARK_CSV,
    },
)

legacy_candidates = [
    LEGACY_FORECAST_CSV,
    resolved_data_config.get("benchmark_csv"),
    resolved_data_config.get("forecast_csv"),
    (resolved_data_config.get("aliases") or {}).get("legacy_prediction_csv"),
]
LEGACY_FORECAST_SOURCE = next((candidate for candidate in legacy_candidates if candidate), None)

expected_paths = [
    ROOT / "Makefile",
    ROOT / "scripts" / "train_consumption.py",
    ROOT / "scripts" / "benchmark_replay_models.py",
    ROOT / "scripts" / "predict_next_day.py",
    ROOT / "scripts" / "promote_consumption_run.py",
] + [
    Path(path).resolve() if Path(path).is_absolute() else (ROOT / path).resolve()
    for path in CONFIG_PATHS
]

checks_df = pd.DataFrame(
    [{"path": str(path), "exists": path.exists()} for path in expected_paths]
)
display(checks_df)

if not checks_df["exists"].all():
    raise FileNotFoundError("Un ou plusieurs fichiers critiques du notebook sont absents.")

dataset_view = pd.DataFrame(
    {
        "dataset_key": [resolved_data_config.get("dataset_key")],
        "catalog_path": [resolved_data_config.get("catalog_path")],
        "historical_csv": [resolved_data_config.get("historical_csv")],
        "forecast_csv": [resolved_data_config.get("forecast_csv")],
        "benchmark_csv": [resolved_data_config.get("benchmark_csv")],
        "weather_csv": [resolved_data_config.get("weather_csv")],
        "holidays_xlsx": [resolved_data_config.get("holidays_xlsx")],
        "legacy_forecast_source": [LEGACY_FORECAST_SOURCE],
    }
).T
dataset_view.columns = ["value"]
display(dataset_view)

if RUN_ENVIRONMENT_CHECKS:
    _ = run_cli(["make", "help"], cwd=ROOT)
    _ = run_cli(["git", "branch", "--show-current"], cwd=ROOT, check=False)
    _ = run_cli(["git", "status", "--short"], cwd=ROOT, check=False)


,path,exists
0,/home/khalid/cours/m1/SmartGrid_Project/smart-grid/Makefile,True
1,/home/khalid/cours/m1/SmartGrid_Project/smart-grid/scripts/train_consumption.py,True
2,/home/khalid/cours/m1/SmartGrid_Project/smart-grid/scripts/benchmark_replay_models.py,True
3,/home/khalid/cours/m1/SmartGrid_Project/smart-grid/scripts/predict_next_day.py,True
4,/home/khalid/cours/m1/SmartGrid_Project/smart-grid/scripts/promote_consumption_run.py,True
5,configs/consumption/mlp_strict_day_ahead_baseline.yaml,False
6,configs/consumption/mlp_strict_day_ahead_weather_basic.yaml,False
7,configs/consumption/mlp_strict_day_ahead_cyclical_weather_basic.yaml,False
8,configs/consumption/mlp_strict_day_ahead_cyclical_weather_shifted_dynamics.yaml,False


FileNotFoundError: Un ou plusieurs fichiers critiques du notebook sont absents.

## D. Config inventory

Tableau des configs candidates avec leur éligibilité officielle au leaderboard métier.


In [ ]:
config_df = build_config_inventory(ROOT, CONFIG_PATHS)
official_config_df = config_df.loc[config_df["official_eligible"] == True].copy()  # noqa: E712

display(config_df)
print("Official strict day-ahead configs:", len(official_config_df))


## E. Training orchestration via CLI / Make

Cette section lance les entraînements avec le vrai pipeline repo. Aucun entraînement n'est réimplémenté dans le notebook.


In [ ]:
training_payloads = []

if RUN_TRAINING:
    for config_path in official_config_df["config_path"].tolist():
        cmd = [
            "make",
            "train-consumption",
            f"CONFIG={config_path}",
            f"ANALYSIS_DAYS={ANALYSIS_DAYS}",
        ] + make_overrides(
            dataset_key=DATASET_KEY,
            historical_csv=HISTORICAL_CSV,
            weather_csv=WEATHER_CSV,
            holidays_xlsx=HOLIDAYS_XLSX,
            benchmark_csv=BENCHMARK_CSV,
        )
        result = run_cli(cmd, cwd=ROOT)
        payload = result.extract_json()
        payload["config_path"] = config_path
        payload["config_name"] = Path(config_path).stem
        training_payloads.append(payload)

training_df = pd.DataFrame(training_payloads)
display(training_df)


## F. Existing run catalog

On recharge les `run_summary.json` présents dans `artifacts/exports/consumption/` pour reconstruire le catalogue des runs utilisables.


In [ ]:
all_runs_df = collect_consumption_runs(ARTIFACTS) if LOAD_EXISTING_RUNS else pd.DataFrame()
if RUN_TRAINING:
    all_runs_df = collect_consumption_runs(ARTIFACTS)

runs_df = all_runs_df.copy()
if ONLY_STRICT_DAY_AHEAD and not runs_df.empty:
    runs_df = runs_df.loc[runs_df["official_eligible"] == True].copy()  # noqa: E712

latest_run_ids_for_configs = select_latest_runs_per_config(
    runs_df,
    CONFIG_PATHS,
    root=ROOT,
    official_only=ONLY_STRICT_DAY_AHEAD,
)

display(
    runs_df[
        [
            col
            for col in [
                "run_id",
                "human_label",
                "config_name",
                "forecast_mode",
                "dataset_key",
                "offline_MAE",
                "offline_RMSE",
                "offline_InTolerance%",
                "summary_json",
            ]
            if col in runs_df.columns
        ]
    ]
)
print("Latest candidate run_ids for configured YAMLs:", latest_run_ids_for_configs)


## G. Replay orchestration and official leaderboard

Le replay est la source de vérité pour le classement métier. Les métriques offline restent annexes.


In [ ]:
replay_payload = None
replay_source = None
replay_summary_df = pd.DataFrame()

if REPLAY_MODEL_RUN_IDS:
    replay_candidate_run_ids = [str(run_id) for run_id in REPLAY_MODEL_RUN_IDS]
elif not training_df.empty:
    replay_candidate_run_ids = training_df["run_id"].astype(str).tolist()
else:
    replay_candidate_run_ids = latest_run_ids_for_configs

print("Replay candidate run_ids:", replay_candidate_run_ids)

if RUN_REPLAY and replay_candidate_run_ids:
    cmd = [
        "python",
        "scripts/benchmark_replay_models.py",
        "--start-date",
        REPLAY_START_DATE,
        "--end-date",
        REPLAY_END_DATE,
    ] + optional_cli_args(
        dataset_key=DATASET_KEY,
        catalog_path=CATALOG_PATH,
        historical_csv=HISTORICAL_CSV,
        weather_csv=WEATHER_CSV,
        holidays_xlsx=HOLIDAYS_XLSX,
        benchmark_csv=BENCHMARK_CSV,
    ) + replay_candidate_run_ids
    result = run_cli(cmd, cwd=ROOT)
    replay_payload = result.extract_json()
    replay_source = replay_payload["summary_csv"]
    replay_summary_df = load_replay_summary(replay_source)
elif LOAD_LATEST_REPLAY_IF_NOT_RUN:
    latest_replay_summary = find_latest_replay_summary(
        ARTIFACTS,
        start_date=REPLAY_START_DATE,
        end_date=REPLAY_END_DATE,
    )
    if latest_replay_summary is None:
        latest_replay_summary = find_latest_replay_summary(ARTIFACTS)
    if latest_replay_summary is not None:
        replay_source = str(latest_replay_summary.resolve())
        replay_summary_df = load_replay_summary(latest_replay_summary)

replay_leaderboard_df = normalize_replay_summary(replay_summary_df, runs_df=runs_df)
if ONLY_STRICT_DAY_AHEAD and not replay_leaderboard_df.empty:
    replay_leaderboard_df = replay_leaderboard_df.loc[
        replay_leaderboard_df["official_eligible"] == True  # noqa: E712
    ].reset_index(drop=True)

replay_export_path = EXPORT_ROOT / "replay_leaderboard_current.csv"
if not replay_leaderboard_df.empty:
    replay_leaderboard_df.to_csv(replay_export_path, index=False)
    print("Replay leaderboard export:", replay_export_path)
    print("Replay source:", replay_source)

display(replay_leaderboard_df)


## H. Skipped-days audit

Les `skipped_days` doivent être visibles au premier coup d'œil, avec leurs dates et raisons.


In [ ]:
skipped_detail_df, skipped_counts_df = build_skipped_days_audit(replay_leaderboard_df)

coverage_view_df = replay_leaderboard_df[
    [
        col
        for col in [
            "human_label",
            "requested_model_run_id",
            "n_requested_days",
            "n_forecasted_days",
            "n_skipped_days",
            "skip_rate_pct",
            "MAE",
            "RMSE",
        ]
        if col in replay_leaderboard_df.columns
    ]
].copy() if not replay_leaderboard_df.empty else pd.DataFrame()

skipped_detail_export = EXPORT_ROOT / "skipped_days_audit.csv"
skipped_counts_export = EXPORT_ROOT / "skipped_days_counts.csv"
skipped_detail_df.to_csv(skipped_detail_export, index=False)
skipped_counts_df.to_csv(skipped_counts_export, index=False)

display(coverage_view_df)
if skipped_counts_df.empty:
    display(pd.DataFrame({"message": ["No skipped replay days in the currently loaded benchmark."]}))
else:
    display(skipped_counts_df)
    display(skipped_detail_df)

print("Skipped-days detail export:", skipped_detail_export)
print("Skipped-days counts export:", skipped_counts_export)


## I. Active run selection and overlay shortlist

Le run actif pilote par défaut :

1. le gagnant du replay si disponible,
2. sinon le `SELECTED_RUN_ID`,
3. sinon le dernier run officiel détecté.


In [ ]:
if SELECTED_RUN_ID:
    ACTIVE_RUN_ID = str(SELECTED_RUN_ID)
elif not replay_leaderboard_df.empty:
    ACTIVE_RUN_ID = str(replay_leaderboard_df.iloc[0]["requested_model_run_id"])
elif not runs_df.empty:
    ACTIVE_RUN_ID = str(runs_df.iloc[0]["run_id"])
else:
    ACTIVE_RUN_ID = None

if COMPARE_RUN_IDS:
    COMPARE_RUN_IDS_RESOLVED = [str(run_id) for run_id in COMPARE_RUN_IDS]
elif not replay_leaderboard_df.empty:
    COMPARE_RUN_IDS_RESOLVED = (
        replay_leaderboard_df["requested_model_run_id"].astype(str).head(COMPARE_TOP_K).tolist()
    )
elif ACTIVE_RUN_ID:
    COMPARE_RUN_IDS_RESOLVED = [ACTIVE_RUN_ID]
else:
    COMPARE_RUN_IDS_RESOLVED = []

if ACTIVE_RUN_ID and ACTIVE_RUN_ID not in COMPARE_RUN_IDS_RESOLVED:
    COMPARE_RUN_IDS_RESOLVED = [ACTIVE_RUN_ID] + COMPARE_RUN_IDS_RESOLVED

selection_label_map = build_model_label_map(
    COMPARE_RUN_IDS_RESOLVED,
    runs_df=runs_df,
    replay_df=replay_leaderboard_df,
)
selection_df = pd.DataFrame(
    {
        "run_id": COMPARE_RUN_IDS_RESOLVED,
        "label": [selection_label_map.get(run_id, run_id) for run_id in COMPARE_RUN_IDS_RESOLVED],
        "is_active": [run_id == ACTIVE_RUN_ID for run_id in COMPARE_RUN_IDS_RESOLVED],
    }
)

active_run_row = (
    runs_df.loc[runs_df["run_id"].astype(str) == str(ACTIVE_RUN_ID)].head(1)
    if ACTIVE_RUN_ID and not runs_df.empty
    else pd.DataFrame()
)

print("ACTIVE_RUN_ID:", ACTIVE_RUN_ID)
display(selection_df)
if not active_run_row.empty:
    display(active_run_row.T)


## J. Long-sample runtime prediction

Section prioritaire : prédire une plage continue de `2025-11-20` à `2026-03-19` avec la vraie CLI runtime et un cache explicite.


In [ ]:
long_sample_frames = {}
long_sample_status_rows = []

if LONG_SAMPLE_RUN_IDS:
    long_sample_run_ids = [str(run_id) for run_id in LONG_SAMPLE_RUN_IDS]
elif ACTIVE_RUN_ID:
    long_sample_run_ids = [ACTIVE_RUN_ID]
else:
    long_sample_run_ids = []

for run_id in long_sample_run_ids:
    cache_csv = (
        EXPORT_ROOT
        / "long_sample_predict"
        / run_id
        / f"{LONG_SAMPLE_START_DATE}__{LONG_SAMPLE_END_DATE}"
        / "predict_long_sample.csv"
    )
    should_load_or_run = RUN_LONG_SAMPLE_PREDICT or (LOAD_CACHED_LONG_SAMPLE and cache_csv.exists())
    if not should_load_or_run:
        continue

    frame, metadata = load_or_run_long_sample_predictions(
        root=ROOT,
        artifacts_root=ARTIFACTS,
        export_root=EXPORT_ROOT,
        run_id=run_id,
        start_date=LONG_SAMPLE_START_DATE,
        end_date=LONG_SAMPLE_END_DATE,
        dataset_key=DATASET_KEY,
        catalog_path=CATALOG_PATH,
        historical_csv=HISTORICAL_CSV,
        weather_csv=WEATHER_CSV,
        holidays_xlsx=HOLIDAYS_XLSX,
        benchmark_csv=BENCHMARK_CSV,
        allow_fallback=False,
        force_recompute=FORCE_RECOMPUTE_LONG_SAMPLE,
        use_cache=LOAD_CACHED_LONG_SAMPLE,
        continue_on_error=LONG_SAMPLE_CONTINUE_ON_ERROR,
    )
    long_sample_frames[run_id] = frame
    long_sample_status_rows.append(
        {
            "run_id": run_id,
            "label": selection_label_map.get(run_id, run_id),
            "rows": int(len(frame)),
            "start_date": metadata.get("start_date"),
            "end_date": metadata.get("end_date"),
            "n_days_requested": metadata.get("n_days_requested"),
            "n_days_completed": metadata.get("n_days_completed"),
            "n_days_failed": metadata.get("n_days_failed"),
            "output_csv": metadata.get("output_csv"),
        }
    )

long_sample_status_df = pd.DataFrame(long_sample_status_rows)
display(long_sample_status_df)


## K. Replay detail loading and offline diagnostics

On charge ici les CSV détaillés du replay officiel et les backtests offline diagnostics des runs retenus pour l'analyse.


In [ ]:
replay_detail_frames = {}
offline_backtest_frames = {}
detail_status_rows = []

for run_id in COMPARE_RUN_IDS_RESOLVED:
    replay_row = (
        replay_leaderboard_df.loc[
            replay_leaderboard_df["requested_model_run_id"].astype(str) == str(run_id)
        ].head(1)
        if not replay_leaderboard_df.empty
        else pd.DataFrame()
    )
    if not replay_row.empty and Path(replay_row.iloc[0]["replay_csv"]).exists():
        replay_frame = pd.read_csv(replay_row.iloc[0]["replay_csv"])
        replay_detail_frames[run_id] = coerce_datetime(replay_frame, "Date")

    run_row = (
        runs_df.loc[runs_df["run_id"].astype(str) == str(run_id)].head(1)
        if not runs_df.empty
        else pd.DataFrame()
    )
    if not run_row.empty:
        backtest_csv = run_row.iloc[0].get("backtest_csv")
        if backtest_csv and Path(backtest_csv).exists():
            offline_backtest_frames[run_id] = coerce_datetime(pd.read_csv(backtest_csv), "Date")

    detail_status_rows.append(
        {
            "run_id": run_id,
            "label": selection_label_map.get(run_id, run_id),
            "has_replay_detail": run_id in replay_detail_frames,
            "has_offline_backtest": run_id in offline_backtest_frames,
        }
    )

detail_status_df = pd.DataFrame(detail_status_rows)
display(detail_status_df)


## L. Legacy + weekly naive + real data preparation

Cette section prépare la vérité terrain et la baseline weekly naive sur toute la plage historique utile, puis borne explicitement le legacy au `2025-12-10`.


In [ ]:
history_df = load_history(
    resolved_data_config["historical_csv"],
    date_col=resolved_data_config.get("date_col", "Date"),
    target_col=resolved_data_config.get("target_name", "tot"),
)
history_df = coerce_datetime(history_df, "Date")
truth_baseline_df = build_truth_baseline_frame(history_df, date_col="Date", real_col="tot")

legacy_raw_df = (
    load_old_benchmark(LEGACY_FORECAST_SOURCE, date_col="Date")
    if LEGACY_FORECAST_SOURCE
    else pd.DataFrame()
)
legacy_prepared_df = prepare_legacy_forecast_frame(
    legacy_raw_df,
    coverage_end_date=LEGACY_COVERAGE_END_DATE,
)

coverage_df = pd.DataFrame(
    {
        "truth_start": [truth_baseline_df["Date"].min()],
        "truth_end": [truth_baseline_df["Date"].max()],
        "legacy_start": [legacy_prepared_df["Date"].min() if not legacy_prepared_df.empty else None],
        "legacy_end": [legacy_prepared_df["Date"].max() if not legacy_prepared_df.empty else None],
        "legacy_coverage_end_date_rule": [LEGACY_COVERAGE_END_DATE],
    }
).T
coverage_df.columns = ["value"]
display(coverage_df)
display(truth_baseline_df.head())
display(legacy_prepared_df.head())


## M. Comparison plots, joined exports, and multi-model overlays

Cette section produit les exports utiles pour une analyse ultérieure et les graphes principaux de démo.


In [ ]:
def export_frame(frame: pd.DataFrame, relative_path: str) -> Path:
    output_path = EXPORT_ROOT / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_path, index=False)
    return output_path


def comparison_model_columns(frame: pd.DataFrame) -> list[str]:
    ignored = {"Date", "real", "weekly_naive", "legacy_forecast"}
    return [column for column in frame.columns if column not in ignored]


def compute_range_metrics(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model_col in comparison_model_columns(frame):
        valid = frame[["Date", "real", model_col]].dropna().copy()
        if valid.empty:
            continue
        metrics = compute_metrics_v2(
            build_metrics_df(valid.set_index("Date"), real_col="real", fc_col=model_col)
        )
        rows.append({"model": model_col, **metrics})
    return pd.DataFrame(rows).sort_values("MAE") if rows else pd.DataFrame()


def compute_daily_metrics(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    rows = []
    model_cols = comparison_model_columns(frame)
    work = frame.copy()
    work["target_date"] = work["Date"].dt.date.astype(str)
    for target_date, day_frame in work.groupby("target_date"):
        for model_col in model_cols:
            valid = day_frame[["Date", "real", model_col]].dropna().copy()
            if valid.empty:
                continue
            metrics = compute_metrics_v2(
                build_metrics_df(valid.set_index("Date"), real_col="real", fc_col=model_col)
            )
            rows.append({"target_date": target_date, "model": model_col, **metrics})
    return pd.DataFrame(rows).sort_values(["target_date", "MAE"]) if rows else pd.DataFrame()


def plot_overlay(frame: pd.DataFrame, title: str, *, linewidth: float = 1.8) -> None:
    if frame.empty:
        print(f"No data for {title}")
        return
    model_cols = comparison_model_columns(frame)
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(frame["Date"], frame["real"], label="real", linewidth=2.6, color="black")
    if "weekly_naive" in frame.columns:
        ax.plot(
            frame["Date"],
            frame["weekly_naive"],
            label="weekly_naive",
            linestyle="--",
            color="tab:gray",
        )
    if "legacy_forecast" in frame.columns and frame["legacy_forecast"].notna().any():
        ax.plot(
            frame["Date"],
            frame["legacy_forecast"],
            label="legacy_forecast (<= 2025-12-10)",
            linestyle=":",
            color="tab:orange",
        )
    for model_col in model_cols:
        ax.plot(frame["Date"], frame[model_col], label=model_col, linewidth=linewidth)
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Consumption")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_daily_mean_overlay(frame: pd.DataFrame, title: str) -> None:
    if frame.empty:
        print(f"No data for {title}")
        return
    cols = ["real", "weekly_naive", "legacy_forecast"] + comparison_model_columns(frame)
    available_cols = [column for column in cols if column in frame.columns]
    daily = frame.set_index("Date")[available_cols].resample("D").mean().reset_index()
    plot_overlay(daily, title, linewidth=2.0)


runtime_model_frames = {
    run_id: frame for run_id, frame in long_sample_frames.items() if frame is not None and not frame.empty
}
runtime_label_map = build_model_label_map(
    list(runtime_model_frames),
    runs_df=runs_df,
    replay_df=replay_leaderboard_df,
)

long_sample_compare_df = pd.DataFrame()
long_sample_metrics_df = pd.DataFrame()
if runtime_model_frames:
    long_sample_compare_df = build_wide_comparison_frame(
        truth_baseline_df=truth_baseline_df,
        model_frames=runtime_model_frames,
        label_map=runtime_label_map,
        start_date=LONG_SAMPLE_START_DATE,
        end_date=LONG_SAMPLE_END_DATE,
        legacy_df=legacy_prepared_df,
    )
    long_sample_export_path = export_frame(
        long_sample_compare_df,
        f"comparisons/long_sample/model_range_comparison_{LONG_SAMPLE_START_DATE}_{LONG_SAMPLE_END_DATE}.csv",
    )
    long_sample_metrics_df = compute_range_metrics(long_sample_compare_df)
    long_sample_daily_metrics_df = compute_daily_metrics(long_sample_compare_df)
    if not long_sample_daily_metrics_df.empty:
        export_frame(
            long_sample_daily_metrics_df,
            f"comparisons/long_sample/per_day_metrics_{LONG_SAMPLE_START_DATE}_{LONG_SAMPLE_END_DATE}.csv",
        )

    if ACTIVE_RUN_ID in runtime_model_frames:
        active_long_sample_compare_df = build_wide_comparison_frame(
            truth_baseline_df=truth_baseline_df,
            model_frames={ACTIVE_RUN_ID: runtime_model_frames[ACTIVE_RUN_ID]},
            label_map={ACTIVE_RUN_ID: runtime_label_map.get(ACTIVE_RUN_ID, ACTIVE_RUN_ID)},
            start_date=LONG_SAMPLE_START_DATE,
            end_date=LONG_SAMPLE_END_DATE,
            legacy_df=legacy_prepared_df,
        )
        export_frame(
            active_long_sample_compare_df,
            f"long_sample_predict/{ACTIVE_RUN_ID}/{LONG_SAMPLE_START_DATE}__{LONG_SAMPLE_END_DATE}/joined_comparison.csv",
        )

    print("Long-sample joined comparison export:", long_sample_export_path)
    display(long_sample_metrics_df)
    plot_daily_mean_overlay(
        long_sample_compare_df,
        f"Long-sample runtime comparison (daily mean) — {LONG_SAMPLE_START_DATE} to {LONG_SAMPLE_END_DATE}",
    )

    for day in SINGLE_DAY_PLOTS:
        day_frame = slice_single_day(long_sample_compare_df, day)
        if day_frame.empty:
            continue
        day_export_path = export_frame(
            day_frame,
            f"comparisons/days/model_day_comparison_{day}.csv",
        )
        print("Day export:", day_export_path)
        plot_overlay(day_frame, f"Runtime long-sample comparison — {day}")
else:
    print(
        "No long-sample runtime predictions loaded. "
        "Set RUN_LONG_SAMPLE_PREDICT=True or enable LOAD_CACHED_LONG_SAMPLE."
    )

replay_label_map = build_model_label_map(
    list(replay_detail_frames),
    runs_df=runs_df,
    replay_df=replay_leaderboard_df,
)
replay_compare_df = pd.DataFrame()
replay_metrics_df = pd.DataFrame()
if replay_detail_frames:
    replay_compare_df = build_wide_comparison_frame(
        truth_baseline_df=truth_baseline_df,
        model_frames=replay_detail_frames,
        label_map=replay_label_map,
        start_date=REPLAY_START_DATE,
        end_date=REPLAY_END_DATE,
        legacy_df=legacy_prepared_df,
    )
    replay_export_path = export_frame(
        replay_compare_df,
        f"comparisons/replay/model_range_comparison_{REPLAY_START_DATE}_{REPLAY_END_DATE}.csv",
    )
    replay_metrics_df = compute_range_metrics(replay_compare_df)
    print("Replay joined comparison export:", replay_export_path)
    display(replay_metrics_df)
    plot_daily_mean_overlay(
        replay_compare_df,
        f"Official replay comparison (daily mean) — {REPLAY_START_DATE} to {REPLAY_END_DATE}",
    )


## N. Error views for the active model

Erreurs absolues, erreurs signées et métriques par jour pour le run actif. Le support privilégié est le long sample runtime s'il est chargé, sinon le replay officiel.


In [ ]:
active_label = (
    build_model_label_map([ACTIVE_RUN_ID], runs_df=runs_df, replay_df=replay_leaderboard_df).get(ACTIVE_RUN_ID)
    if ACTIVE_RUN_ID
    else None
)

if active_label and not long_sample_compare_df.empty and active_label in long_sample_compare_df.columns:
    error_source_name = "runtime_long_sample"
    error_source_df = long_sample_compare_df.copy()
elif active_label and not replay_compare_df.empty and active_label in replay_compare_df.columns:
    error_source_name = "official_replay"
    error_source_df = replay_compare_df.copy()
else:
    error_source_name = None
    error_source_df = pd.DataFrame()

active_error_df = pd.DataFrame()
active_per_day_metrics_df = pd.DataFrame()

if error_source_name and not error_source_df.empty and active_label:
    active_error_df = error_source_df[["Date", "real", active_label]].dropna().copy()
    active_error_df["error"] = active_error_df[active_label] - active_error_df["real"]
    active_error_df["abs_error"] = active_error_df["error"].abs()
    active_error_df["target_date"] = active_error_df["Date"].dt.date.astype(str)
    active_per_day_metrics_df = compute_daily_metrics(
        error_source_df[["Date", "real", active_label]].rename(columns={active_label: active_label})
    )

    error_export_path = export_frame(
        active_error_df,
        f"comparisons/errors/error_series_{error_source_name}_{ACTIVE_RUN_ID}.csv",
    )
    if not active_per_day_metrics_df.empty:
        export_frame(
            active_per_day_metrics_df,
            f"comparisons/errors/per_day_metrics_{error_source_name}_{ACTIVE_RUN_ID}.csv",
        )

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(active_error_df["Date"], active_error_df["abs_error"], color="tab:red")
    axes[0].set_title(f"Absolute error over time — {error_source_name}")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(active_error_df["Date"], active_error_df["error"], color="tab:blue")
    axes[1].axhline(0.0, color="black", linewidth=1.0)
    axes[1].set_title(f"Signed error over time — {error_source_name}")
    axes[1].grid(True, alpha=0.3)

    axes[2].hist(active_error_df["abs_error"].dropna(), bins=40, color="tab:purple", alpha=0.8)
    axes[2].set_title("Absolute error histogram")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    display(active_per_day_metrics_df.head(20))
    print("Active error export:", error_export_path)
else:
    print("No error source available for the active run yet.")


## O. Protocol comparison audit

Audit technique uniquement. Cette section compare, pour **un run** et **une date**, les trois protocoles :

- `offline diagnostic`,
- `runtime predict`,
- `replay same-day`.

Ce n'est **pas** le leaderboard officiel.


In [ ]:
protocol_predict_df = pd.DataFrame()
protocol_compare_df = pd.DataFrame()
protocol_metrics_df = pd.DataFrame()

if ACTIVE_RUN_ID:
    active_run_row = runs_df.loc[runs_df["run_id"].astype(str) == str(ACTIVE_RUN_ID)].head(1)
    active_run_dir = (
        Path(active_run_row.iloc[0]["run_dir"])
        if not active_run_row.empty
        else ARTIFACTS / "runs" / "consumption" / ACTIVE_RUN_ID
    )
    protocol_output_dir = EXPORT_ROOT / "protocol_audit" / ACTIVE_RUN_ID / PROTOCOL_AUDIT_DATE
    protocol_predict_csv = protocol_output_dir / f"predict_target_day_{PROTOCOL_AUDIT_DATE}.csv"

    if RUN_PROTOCOL_AUDIT or protocol_predict_csv.exists():
        if RUN_PROTOCOL_AUDIT or not protocol_predict_csv.exists():
            protocol_output_dir.mkdir(parents=True, exist_ok=True)
            result = run_cli(
                [
                    "python",
                    "scripts/predict_next_day.py",
                    "--current-dir",
                    str(active_run_dir),
                    "--target-date",
                    PROTOCOL_AUDIT_DATE,
                    "--output-csv",
                    str(protocol_predict_csv),
                ]
                + optional_cli_args(
                    dataset_key=DATASET_KEY,
                    catalog_path=CATALOG_PATH,
                    historical_csv=HISTORICAL_CSV,
                    weather_csv=WEATHER_CSV,
                    holidays_xlsx=HOLIDAYS_XLSX,
                    benchmark_csv=BENCHMARK_CSV,
                ),
                cwd=ROOT,
            )
            _ = result.extract_json()
        protocol_predict_df = coerce_datetime(pd.read_csv(protocol_predict_csv), "Date")

    offline_day_df = slice_single_day(
        offline_backtest_frames.get(ACTIVE_RUN_ID, pd.DataFrame()),
        PROTOCOL_AUDIT_DATE,
    )
    replay_day_df = slice_single_day(
        replay_detail_frames.get(ACTIVE_RUN_ID, pd.DataFrame()),
        PROTOCOL_AUDIT_DATE,
    )
    truth_day_df = slice_single_day(truth_baseline_df, PROTOCOL_AUDIT_DATE)[
        ["Date", "real", "weekly_naive"]
    ]
    legacy_day_df = slice_single_day(legacy_prepared_df, PROTOCOL_AUDIT_DATE)

    protocol_compare_df = truth_day_df.copy()
    if not legacy_day_df.empty:
        protocol_compare_df = protocol_compare_df.merge(legacy_day_df, on="Date", how="left")
    else:
        protocol_compare_df["legacy_forecast"] = pd.NA

    if not offline_day_df.empty and "Ptot_TOTAL_Forecast" in offline_day_df.columns:
        protocol_compare_df = protocol_compare_df.merge(
            offline_day_df[["Date", "Ptot_TOTAL_Forecast"]].rename(
                columns={"Ptot_TOTAL_Forecast": "offline_diagnostic"}
            ),
            on="Date",
            how="left",
        )
    if not protocol_predict_df.empty and "Ptot_TOTAL_Forecast" in protocol_predict_df.columns:
        protocol_compare_df = protocol_compare_df.merge(
            protocol_predict_df[["Date", "Ptot_TOTAL_Forecast"]].rename(
                columns={"Ptot_TOTAL_Forecast": "runtime_predict"}
            ),
            on="Date",
            how="left",
        )
    if not replay_day_df.empty and "Ptot_TOTAL_Forecast" in replay_day_df.columns:
        protocol_compare_df = protocol_compare_df.merge(
            replay_day_df[["Date", "Ptot_TOTAL_Forecast"]].rename(
                columns={"Ptot_TOTAL_Forecast": "replay_same_day"}
            ),
            on="Date",
            how="left",
        )

    protocol_compare_df = coerce_datetime(protocol_compare_df, "Date")
    protocol_compare_export_path = export_frame(
        protocol_compare_df,
        f"protocol_audit/{ACTIVE_RUN_ID}/model_day_comparison_{PROTOCOL_AUDIT_DATE}.csv",
    )
    protocol_model_cols = [
        column
        for column in ["offline_diagnostic", "runtime_predict", "replay_same_day"]
        if column in protocol_compare_df.columns
    ]
    rows = []
    for column in protocol_model_cols:
        valid = protocol_compare_df[["Date", "real", column]].dropna().copy()
        if valid.empty:
            continue
        metrics = compute_metrics_v2(
            build_metrics_df(valid.set_index("Date"), real_col="real", fc_col=column)
        )
        rows.append({"protocol": column, **metrics})
    protocol_metrics_df = pd.DataFrame(rows).sort_values("MAE") if rows else pd.DataFrame()

    plot_cols = [
        column
        for column in [
            "Date",
            "real",
            "weekly_naive",
            "legacy_forecast",
            "offline_diagnostic",
            "runtime_predict",
            "replay_same_day",
        ]
        if column in protocol_compare_df.columns
    ]
    if len(plot_cols) > 1:
        plot_overlay(
            protocol_compare_df[plot_cols],
            f"Technical parity audit — {ACTIVE_RUN_ID} — {PROTOCOL_AUDIT_DATE}",
        )

    display(protocol_metrics_df)
    print("Protocol audit export:", protocol_compare_export_path)
else:
    print("No active run selected.")


## P. Final ranking, plots, and notebook metadata

Classement final **replay-first**, enrichi avec la couverture et les métriques offline en colonne de contexte.


In [ ]:
final_ranking_df = replay_leaderboard_df.copy()
if ONLY_STRICT_DAY_AHEAD and not final_ranking_df.empty:
    final_ranking_df = final_ranking_df.loc[
        final_ranking_df["official_eligible"] == True  # noqa: E712
    ].reset_index(drop=True)

ranking_cols = [
    column
    for column in [
        "human_label",
        "requested_model_run_id",
        "config_name",
        "MAE",
        "RMSE",
        "RampingError_RMSE",
        "n_requested_days",
        "n_forecasted_days",
        "n_skipped_days",
        "skip_rate_pct",
        "offline_MAE",
        "offline_RMSE",
        "offline_InTolerance%",
    ]
    if column in final_ranking_df.columns
]
display(final_ranking_df[ranking_cols] if not final_ranking_df.empty else final_ranking_df)

if not final_ranking_df.empty:
    labels = final_ranking_df["human_label"].astype(str).tolist()
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    axes[0, 0].barh(labels, final_ranking_df["MAE"], color="tab:blue")
    axes[0, 0].invert_yaxis()
    axes[0, 0].set_title("Replay MAE ranking")
    axes[0, 0].grid(True, axis="x", alpha=0.3)

    axes[0, 1].barh(labels, final_ranking_df["RMSE"], color="tab:green")
    axes[0, 1].invert_yaxis()
    axes[0, 1].set_title("Replay RMSE ranking")
    axes[0, 1].grid(True, axis="x", alpha=0.3)

    if "skip_rate_pct" in final_ranking_df.columns:
        axes[1, 0].scatter(final_ranking_df["skip_rate_pct"], final_ranking_df["MAE"], color="tab:red")
        for _, row in final_ranking_df.iterrows():
            axes[1, 0].annotate(str(row["human_label"]), (row["skip_rate_pct"], row["MAE"]))
        axes[1, 0].set_title("MAE vs skip rate")
        axes[1, 0].set_xlabel("skip_rate_pct")
        axes[1, 0].set_ylabel("MAE")
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].set_visible(False)

    if "RampingError_RMSE" in final_ranking_df.columns:
        axes[1, 1].scatter(
            final_ranking_df["RampingError_RMSE"],
            final_ranking_df["MAE"],
            color="tab:purple",
        )
        for _, row in final_ranking_df.iterrows():
            axes[1, 1].annotate(str(row["human_label"]), (row["RampingError_RMSE"], row["MAE"]))
        axes[1, 1].set_title("MAE vs RampingError_RMSE")
        axes[1, 1].set_xlabel("RampingError_RMSE")
        axes[1, 1].set_ylabel("MAE")
        axes[1, 1].grid(True, alpha=0.3)
    else:
        axes[1, 1].set_visible(False)

    plt.tight_layout()
    plt.show()

winner_run_id = (
    str(final_ranking_df.iloc[0]["requested_model_run_id"])
    if not final_ranking_df.empty
    else None
)
winner_label = (
    str(final_ranking_df.iloc[0]["human_label"])
    if not final_ranking_df.empty
    else None
)

metadata_payload = {
    "generated_at": pd.Timestamp.utcnow().isoformat(),
    "active_run_id": ACTIVE_RUN_ID,
    "winner_run_id": winner_run_id,
    "winner_label": winner_label,
    "replay_window": {
        "start_date": REPLAY_START_DATE,
        "end_date": REPLAY_END_DATE,
        "source_summary_csv": replay_source,
    },
    "long_sample_window": {
        "start_date": LONG_SAMPLE_START_DATE,
        "end_date": LONG_SAMPLE_END_DATE,
    },
    "legacy_coverage_end_date": LEGACY_COVERAGE_END_DATE,
    "config_paths": official_config_df["config_path"].tolist() if not official_config_df.empty else list(CONFIG_PATHS),
    "export_root": str(EXPORT_ROOT.resolve()),
}
metadata_path = write_json(EXPORT_ROOT / "metadata.json", metadata_payload)
print("Notebook metadata export:", metadata_path)
print("Replay leaderboard export:", EXPORT_ROOT / "replay_leaderboard_current.csv")


## Q. Promotion

Promotion optionnelle du vainqueur du replay via la CLI officielle. Garder `PROMOTE_REPLAY_WINNER = False` tant que le classement n'a pas été validé.


In [ ]:
promotion_candidate_run_id = (
    str(final_ranking_df.iloc[0]["requested_model_run_id"])
    if not final_ranking_df.empty
    else None
)
print("promotion_candidate_run_id:", promotion_candidate_run_id)

if RUN_PROMOTION and PROMOTE_REPLAY_WINNER and promotion_candidate_run_id:
    _ = run_cli(
        [
            "python",
            "scripts/promote_consumption_run.py",
            "--run-id",
            promotion_candidate_run_id,
        ],
        cwd=ROOT,
    )
else:
    print(
        "Promotion not executed. Set RUN_PROMOTION=True and PROMOTE_REPLAY_WINNER=True "
        "to promote the replay winner."
    )


## R. Final notes

Ce notebook est conçu pour raconter clairement trois niveaux de vérité :

- **replay** = benchmark métier officiel,
- **offline** = diagnostic secondaire,
- **long sample runtime** = démonstration de la chaîne réelle sur une plage continue.

Exports principaux produits sous `artifacts/notebook_exports/cli_demo_v3/` :

- `replay_leaderboard_current.csv`
- `skipped_days_audit.csv`
- `long_sample_predict/<run_id>/<start>__<end>/predict_long_sample.csv`
- `long_sample_predict/<run_id>/<start>__<end>/joined_comparison.csv`
- `comparisons/days/model_day_comparison_<date>.csv`
- `comparisons/replay/model_range_comparison_<start>_<end>.csv`
- `comparisons/long_sample/model_range_comparison_<start>_<end>.csv`
- `metadata.json`
